In [1]:
import pandas as pd
import os

# relative directory path
directory = './data/demand'
dfs = [] 

for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        # Create the full file path and read the file
        file_path = os.path.join(directory, filename)
        df = pd.read_csv(file_path)
        
        # Check if column exists to prevent errors
        print(f"File: {filename}")
        dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df.drop(columns=['REGION', 'PERIODTYPE'], inplace=True)
df.rename(columns={'SETTLEMENTDATE': 'DATE_TIME', 'TOTALDEMAND': 'TOTAL_DEMAND'}, inplace=True, errors='ignore')


File: PRICE_AND_DEMAND_202301_NSW1.csv
File: PRICE_AND_DEMAND_202401_NSW1.csv
File: PRICE_AND_DEMAND_202501_NSW1.csv
File: PRICE_AND_DEMAND_202601_NSW1.csv


In [2]:
print(df.shape)
display(df.head())
display(df.describe())

(35712, 3)


,DATE_TIME,TOTAL_DEMAND,RRP
0,2023/01/01 00:05:00,6906.63,140.40
1,2023/01/01 00:10:00,6854.28,133.96
2,2023/01/01 00:15:00,6856.32,120.00
3,2023/01/01 00:20:00,6796.27,118.99
4,2023/01/01 00:25:00,6805.55,118.99


,TOTAL_DEMAND,RRP
count,35712.000000,35712.000000
mean,7389.831917,81.531417
std,1303.476356,330.302313
min,4015.440000,-999.990000
25%,6503.587500,44.610000
50%,7206.895000,69.700000
75%,8053.817500,97.662500
max,13146.790000,17500.000000


In [3]:
## CREATING COLUMNS FOR SCHOOL_HOLIDAY, PUBLIC_HOLIDAY
import holidays

# Convert to Pandas datetime for better parsing
df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'])

nsw_holidays = holidays.Australia(subdiv='NSW', years=range(2023, 2027))
df['PUBLIC_HOLIDAY'] = df['DATE_TIME'].dt.date.isin(nsw_holidays)

school_holiday_ranges = [
    # 2023
    ('2023-01-01', '2023-01-26'), # Summer 22-23
    ('2023-04-10', '2023-04-21'), # Autumn
    ('2023-07-03', '2023-07-14'), # Winter
    ('2023-09-25', '2023-10-06'), # Spring
    ('2023-12-20', '2023-12-31'), # Summer 23-24 start

    # 2024
    ('2024-01-01', '2024-01-29'), # Summer 23-24 end
    ('2024-04-15', '2024-04-26'), # Autumn
    ('2024-07-08', '2024-07-19'), # Winter
    ('2024-09-30', '2024-10-11'), # Spring
    ('2023-12-23', '2023-12-31'), # Summer 24-25 start

    # 2025
    ('2025-01-01', '2025-01-30'), # Summer 24-25 end
    ('2025-04-14', '2025-04-24'), # Autumn
    ('2025-07-07', '2025-07-18'), # Winter
    ('2025-09-29', '2025-10-10'), # Spring
    ('2025-12-22', '2025-12-31'), # Summer 25-26 start

    # 2026
    ('2026-01-01', '2026-01-26'), # Summer 25-26 end
    ('2026-04-07', '2026-04-17'), # Autumn
    ('2026-07-06', '2026-07-17'), # Winter
    ('2026-09-28', '2026-10-09'), # Spring
    ('2026-12-18', '2026-12-31'), # Summer 26-27 start
]

holiday_dates = set()
for start, end in school_holiday_ranges:
    holiday_dates.update(pd.date_range(start, end).date)

df['SCHOOL_HOLIDAY'] = df['DATE_TIME'].dt.date.isin(holiday_dates)

df.head()

,DATE_TIME,TOTAL_DEMAND,RRP,PUBLIC_HOLIDAY,SCHOOL_HOLIDAY
0,2023-01-01 00:05:00,6906.63,140.40,True,True
1,2023-01-01 00:10:00,6854.28,133.96,True,True
2,2023-01-01 00:15:00,6856.32,120.00,True,True
3,2023-01-01 00:20:00,6796.27,118.99,True,True
4,2023-01-01 00:25:00,6805.55,118.99,True,True


In [4]:
import pandas as pd
import datetime

def fetch_sydney_weather(start_year, end_year):
    # Sydney Airport Station ID is usually 'YSSY' in the ASOS network
    station = 'YSSY' 
    start_date = f"{start_year}-01-01+00:00"
    end_date = f"{end_year}-12-31+23:59"
    
    # Web scraping the global archive - added 'feel' and 'p01m' for transpiration context
    url = (f"https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
           f"station={station}&data=tmpc&data=dwpc&data=relh&data=feel&data=sknt&"
           f"year1={start_year}&month1=1&day1=1&"
           f"year2={end_year}&month2=12&day2=31&"
           f"tz=Etc%2FGMT-10&format=onlycomma&latlon=no&missing=M")
    
    print("Fetching weather data")
    weather_df = pd.read_csv(url, skiprows=0, na_values='M')
    
    # Rename columns for clarity
    weather_df = weather_df.rename(columns={
        'valid': 'DATE_TIME',
        'tmpc': 'TEMPERATURE',
        'relh': 'HUMIDITY',
        'dwpc': 'DWPC',
        'feel': 'APPARENT_TEMP',
        'p01m': 'PRECIPITATION',
        'sknt': 'WIND_SPEED'
    })
    
    weather_df['DATE_TIME'] = pd.to_datetime(weather_df['DATE_TIME'])
    weather_df['APPARENT_TEMP'] = (weather_df['APPARENT_TEMP'] - 32) * 5/9 # Farenheit to Celcius
    weather_df['WIND_SPEED'] = weather_df['WIND_SPEED'] * 1.852 # Convert Wind Speed from knots to km/h (1 knot ≈ 1.852 km/h)
    
    return weather_df

weather_df = fetch_sydney_weather(2023, 2025)
weather_df.drop('station', axis=1, inplace=True, errors='ignore')
display(weather_df.head())


Fetching weather data


,DATE_TIME,TEMPERATURE,DWPC,HUMIDITY,APPARENT_TEMP,WIND_SPEED
0,2023-01-01 00:00:00,22.0,18.0,78.04,22.0,14.816
1,2023-01-01 00:30:00,22.0,18.0,78.04,22.0,14.816
2,2023-01-01 01:00:00,22.0,19.0,83.09,22.0,18.520
3,2023-01-01 01:30:00,22.0,19.0,83.09,22.0,16.668
4,2023-01-01 02:30:00,21.0,18.0,82.98,21.0,12.964


In [5]:
if 'DATE_TIME' in df.columns:
    demand_df = df.set_index('DATE_TIME').sort_index()
else:
    demand_df = df.sort_index()

if 'DATE_TIME' in weather_df.columns:
    weather_df = weather_df.set_index('DATE_TIME').sort_index()
else:
    weather_df = weather_df.sort_index()

weather_numeric = weather_df.select_dtypes(include=['number', 'float', 'int'])
weather_5min = weather_numeric.reindex(demand_df.index)

# Interpolate the gaps (Linear interpolation)
weather_5min = weather_5min.interpolate(method='linear')
weather_5min = weather_5min.bfill().ffill()

# merge back together
demand_df = demand_df.drop(columns=weather_5min.columns.intersection(demand_df.columns))
df = pd.concat([demand_df, weather_5min], axis=1).reset_index()

if 'index' in df.columns:
    df = df.rename(columns={'index': 'DATE_TIME'})
df.head()

,DATE_TIME,TOTAL_DEMAND,RRP,PUBLIC_HOLIDAY,SCHOOL_HOLIDAY,TEMPERATURE,DWPC,HUMIDITY,APPARENT_TEMP,WIND_SPEED
0,2023-01-01 00:05:00,6906.63,140.40,True,True,22.0,18.0,78.04,22.0,14.816
1,2023-01-01 00:10:00,6854.28,133.96,True,True,22.0,18.0,78.04,22.0,14.816
2,2023-01-01 00:15:00,6856.32,120.00,True,True,22.0,18.0,78.04,22.0,14.816
3,2023-01-01 00:20:00,6796.27,118.99,True,True,22.0,18.0,78.04,22.0,14.816
4,2023-01-01 00:25:00,6805.55,118.99,True,True,22.0,18.0,78.04,22.0,14.816


In [6]:
df.DATE_TIME.max()

Timestamp('2026-02-01 00:00:00')

In [7]:
### Consumer spending and confidence


## RBA Monthly Cash Rate
cash_rate_df = pd.read_csv('./data/rba_monthly_cash_rate.csv')
df['Month_Year'] = df['DATE_TIME'].dt.strftime('%Y-%m') # parse date format like '2023-01'
df = df.merge(cash_rate_df, left_on='Month_Year', right_on='DATE', how='left') # broadcast join
df = df.drop(columns=['Month_Year', 'DATE']) # clean columns

## Monthly Household Spending Indicator (MHSI)
spending_df = pd.read_csv('./data/spending_millions.csv')
df['join_key'] = df['DATE_TIME'].dt.strftime('%b-%Y')

# Merge and clean up
df = df.merge(spending_df, left_on='join_key', right_on='Month', how='left')
df = df.drop(columns=['join_key', 'Month'])

# Consumer Confidence (Westpac)
consumer_conf = pd.read_csv('./data/consumer_conf.csv')
df['month_key'] = df['DATE_TIME'].dt.strftime('%Y-%m')
df = df.merge(consumer_conf, left_on='month_key', right_on='year-month', how='left')
df = df.drop(columns=['month_key', 'year-month'])

df.head()

,DATE_TIME,TOTAL_DEMAND,RRP,PUBLIC_HOLIDAY,SCHOOL_HOLIDAY,TEMPERATURE,DWPC,HUMIDITY,APPARENT_TEMP,WIND_SPEED,CASH_RATE,HOUSEHOLD_SPEND,SPEND_GROWTH,CONF_IDX
0,2023-01-01 00:05:00,6906.63,140.40,True,True,22.0,18.0,78.04,22.0,14.816,3.1,21629.7,-13.4,86.9
1,2023-01-01 00:10:00,6854.28,133.96,True,True,22.0,18.0,78.04,22.0,14.816,3.1,21629.7,-13.4,86.9
2,2023-01-01 00:15:00,6856.32,120.00,True,True,22.0,18.0,78.04,22.0,14.816,3.1,21629.7,-13.4,86.9
3,2023-01-01 00:20:00,6796.27,118.99,True,True,22.0,18.0,78.04,22.0,14.816,3.1,21629.7,-13.4,86.9
4,2023-01-01 00:25:00,6805.55,118.99,True,True,22.0,18.0,78.04,22.0,14.816,3.1,21629.7,-13.4,86.9
